# MOSAIC — K2 IA Random Forest + XGBoost

This notebook evaluates the **K2 intercept + amplitude (IA)** spectral information domain using two nonlinear supervised models:

- Random Forest
- XGBoost

For each functional group, the ecological response is decomposed into:

1. **Occurrence**: `presence = cover > 0`
2. **Positive abundance**: `cover | cover > 0`

The objective is **feature-space evaluation**, not replacement of the Bayesian ZIB as the primary inferential model.

Primary outputs:

- Cross-validated occurrence AUC / log loss
- Cross-validated positive-abundance R² / RMSE
- Combined hurdle-style expected-cover R²
- Fold-wise held-out permutation importance
- Importance stability across folds
- Spectral-family summaries for correlated predictors

## Expected CSV exported from R

Export one analysis table containing:

- `PlotID`
- `Year`
- the seven functional-group cover responses
- all 45 K2 IA predictors:
  - 15 `constant_*`
  - 15 `amp1_*`
  - 15 `amp2_*`

Using the same standardized predictor table as the ZIB analysis keeps the feature space directly aligned with the Bayesian work. Tree models themselves do not require standardization.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    brier_score_loss,
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn.model_selection import StratifiedKFold

try:
    from xgboost import XGBClassifier, XGBRegressor
except ImportError as e:
    raise ImportError(
        "xgboost is required. Install it with: pip install xgboost"
    ) from e

RANDOM_STATE = 42
N_SPLITS = 5

DATA_CSV = Path(r"C:/NCA_DATA/Analysis/MOSAIC_K2_IA_ZIB_export.csv")

FUNCTIONAL_GROUPS = [
    "BAREGROUND_FG",
    "BIOCRUST_FG",
    "EAG_FG",
    "EF_FG",
    "LITTER_FG",
    "PBG_FG",
    "SHRUB_total",
]

In [ ]:
# Load and validate the R export

df = pd.read_csv(DATA_CSV)

if "Plot" in df.columns and "PlotID" not in df.columns:
    df = df.rename(columns={"Plot": "PlotID"})

if "PlotID" in df.columns:
    df["PlotID"] = (
        df["PlotID"]
        .astype(str)
        .str.strip()
        .str.replace(r"\\.0$", "", regex=True)
    )

IA_COLS = [
    c for c in df.columns
    if c.lower().startswith((
        "constant_", "intercept_",
        "amp1_", "amp2_",
        "amplitude1_", "amplitude2_"
    ))
]

INTERCEPT_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("constant_", "intercept_"))
]

AMP1_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("amp1_", "amplitude1_"))
]

AMP2_COLS = [
    c for c in IA_COLS
    if c.lower().startswith(("amp2_", "amplitude2_"))
]

print(f"Rows: {len(df):,}")
print(f"IA predictors: {len(IA_COLS)}")
print(f"  intercepts: {len(INTERCEPT_COLS)}")
print(f"  H1 amplitudes: {len(AMP1_COLS)}")
print(f"  H2 amplitudes: {len(AMP2_COLS)}")

missing_fg = [fg for fg in FUNCTIONAL_GROUPS if fg not in df.columns]
if missing_fg:
    raise ValueError(f"Missing functional-group columns: {missing_fg}")

if len(IA_COLS) != 45:
    print("WARNING: expected 45 K2 IA predictors. Inspect IA_COLS before proceeding.")

display(df[["PlotID", "Year"] + FUNCTIONAL_GROUPS + IA_COLS[:5]].head())

## Modeling logic

For functional-group cover \(Y\):

### Occurrence
\[
Z = \mathbb{1}(Y > 0)
\]

A classifier estimates:

\[
\hat p_i = P(Y_i > 0 \mid X_i)
\]

### Positive abundance
For observations with \(Y>0\), a regressor estimates:

\[
\hat \mu_i = E(Y_i \mid Y_i>0, X_i)
\]

### Combined expected cover
For every observation:

\[
\widehat{E(Y_i \mid X_i)} = \hat p_i \hat \mu_i
\]

This mirrors the ecological decomposition used in the hurdle/ZIB work while allowing nonlinear effects and interactions.

In [ ]:
MODELS = {
    "RandomForest": {
        "classifier": RandomForestClassifier(
            n_estimators=800,
            max_features="sqrt",
            min_samples_leaf=3,
            class_weight="balanced",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "regressor": RandomForestRegressor(
            n_estimators=800,
            max_features="sqrt",
            min_samples_leaf=3,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    },

    "XGBoost": {
        "classifier": XGBClassifier(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "regressor": XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.05,
            reg_lambda=1.0,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    },
}

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


def fit_fg_cv(
    data,
    fg,
    feature_cols,
    model_name,
    model_spec,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
    permutation_repeats=15,
):
    cols = feature_cols + [fg]
    work = data[cols].dropna().copy()

    X = work[feature_cols].to_numpy()
    y = work[fg].to_numpy(dtype=float)

    # Accept either percent cover or proportion.
    if np.nanmax(y) > 1.0:
        y = y / 100.0

    presence = (y > 0).astype(int)

    n_pos = int(presence.sum())
    n_zero = int(len(presence) - n_pos)

    if n_pos < n_splits or n_zero < n_splits:
        raise ValueError(
            f"{fg}: insufficient positive/zero observations for {n_splits}-fold CV."
        )

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    oof_p_presence = np.full(len(work), np.nan)
    oof_mu_positive = np.full(len(work), np.nan)

    fold_metrics = []
    importance_rows = []

    for fold, (train_idx, test_idx) in enumerate(cv.split(X, presence), start=1):

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        z_train, z_test = presence[train_idx], presence[test_idx]

        # -------------------------------------------------------------
        # Occurrence
        # -------------------------------------------------------------
        clf = clone(model_spec["classifier"])
        clf.fit(X_train, z_train)

        p_test = clf.predict_proba(X_test)[:, 1]
        oof_p_presence[test_idx] = p_test

        auc = roc_auc_score(z_test, p_test)
        ll = log_loss(z_test, p_test, labels=[0, 1])
        brier = brier_score_loss(z_test, p_test)

        pi_occ = permutation_importance(
            clf,
            X_test,
            z_test,
            scoring="roc_auc",
            n_repeats=permutation_repeats,
            random_state=random_state + fold,
            n_jobs=-1,
        )

        for feature, imp_mean, imp_sd in zip(
            feature_cols,
            pi_occ.importances_mean,
            pi_occ.importances_std,
        ):
            importance_rows.append({
                "functional_group": fg,
                "model": model_name,
                "fold": fold,
                "process": "occurrence",
                "feature": feature,
                "importance_mean": imp_mean,
                "importance_sd": imp_sd,
            })

        # -------------------------------------------------------------
        # Positive abundance
        # -------------------------------------------------------------
        pos_train = y_train > 0
        pos_test = y_test > 0

        reg = clone(model_spec["regressor"])
        reg.fit(X_train[pos_train], y_train[pos_train])

        # Predict conditional abundance for all held-out rows so the
        # two components can be recombined as expected cover.
        mu_test_all = np.clip(reg.predict(X_test), 0.0, 1.0)
        oof_mu_positive[test_idx] = mu_test_all

        if pos_test.sum() >= 2:
            y_pos = y_test[pos_test]
            mu_pos = mu_test_all[pos_test]

            pos_r2 = r2_score(y_pos, mu_pos)
            pos_rmse = rmse(y_pos, mu_pos)
            pos_mae = mean_absolute_error(y_pos, mu_pos)

            pi_ab = permutation_importance(
                reg,
                X_test[pos_test],
                y_pos,
                scoring="r2",
                n_repeats=permutation_repeats,
                random_state=random_state + 100 + fold,
                n_jobs=-1,
            )

            for feature, imp_mean, imp_sd in zip(
                feature_cols,
                pi_ab.importances_mean,
                pi_ab.importances_std,
            ):
                importance_rows.append({
                    "functional_group": fg,
                    "model": model_name,
                    "fold": fold,
                    "process": "abundance",
                    "feature": feature,
                    "importance_mean": imp_mean,
                    "importance_sd": imp_sd,
                })
        else:
            pos_r2 = np.nan
            pos_rmse = np.nan
            pos_mae = np.nan

        expected_cover = p_test * mu_test_all

        fold_metrics.append({
            "functional_group": fg,
            "model": model_name,
            "fold": fold,
            "n_test": len(test_idx),
            "n_positive_test": int(pos_test.sum()),
            "occurrence_auc": auc,
            "occurrence_logloss": ll,
            "occurrence_brier": brier,
            "positive_r2": pos_r2,
            "positive_rmse": pos_rmse,
            "positive_mae": pos_mae,
            "hurdle_r2": r2_score(y_test, expected_cover),
            "hurdle_rmse": rmse(y_test, expected_cover),
        })

    expected_oof = oof_p_presence * oof_mu_positive
    pos_all = y > 0

    summary = {
        "functional_group": fg,
        "model": model_name,
        "n": len(work),
        "n_positive": int(pos_all.sum()),
        "zero_fraction": float((~pos_all).mean()),
        "oof_occurrence_auc": roc_auc_score(presence, oof_p_presence),
        "oof_occurrence_logloss": log_loss(
            presence, oof_p_presence, labels=[0, 1]
        ),
        "oof_occurrence_brier": brier_score_loss(
            presence, oof_p_presence
        ),
        "oof_positive_r2": r2_score(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_positive_rmse": rmse(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_positive_mae": mean_absolute_error(
            y[pos_all], oof_mu_positive[pos_all]
        ),
        "oof_hurdle_r2": r2_score(y, expected_oof),
        "oof_hurdle_rmse": rmse(y, expected_oof),
    }

    predictions = pd.DataFrame({
        "row_index": work.index,
        "cover": y,
        "presence": presence,
        "p_presence_oof": oof_p_presence,
        "mu_positive_oof": oof_mu_positive,
        "expected_cover_oof": expected_oof,
    })

    return (
        pd.DataFrame([summary]),
        pd.DataFrame(fold_metrics),
        pd.DataFrame(importance_rows),
        predictions,
    )

In [ ]:
all_summary = []
all_fold_metrics = []
all_importance = []
all_predictions = []

for fg in FUNCTIONAL_GROUPS:
    for model_name, model_spec in MODELS.items():

        print(f"Running {model_name}: {fg}")

        summary_i, folds_i, importance_i, preds_i = fit_fg_cv(
            data=df,
            fg=fg,
            feature_cols=IA_COLS,
            model_name=model_name,
            model_spec=model_spec,
        )

        preds_i["functional_group"] = fg
        preds_i["model"] = model_name

        all_summary.append(summary_i)
        all_fold_metrics.append(folds_i)
        all_importance.append(importance_i)
        all_predictions.append(preds_i)

summary_results = pd.concat(all_summary, ignore_index=True)
fold_results = pd.concat(all_fold_metrics, ignore_index=True)
importance_results = pd.concat(all_importance, ignore_index=True)
prediction_results = pd.concat(all_predictions, ignore_index=True)

display(
    summary_results.sort_values(
        ["functional_group", "model"]
    )
)

In [ ]:
importance_summary = (
    importance_results
    .groupby(
        ["functional_group", "model", "process", "feature"],
        as_index=False
    )
    .agg(
        mean_importance=("importance_mean", "mean"),
        sd_importance=("importance_mean", "std"),
        median_importance=("importance_mean", "median"),
        positive_fold_fraction=(
            "importance_mean",
            lambda x: np.mean(np.asarray(x) > 0)
        ),
        n_folds=("fold", "nunique"),
    )
)

importance_summary["stability_score"] = (
    importance_summary["mean_importance"]
    * importance_summary["positive_fold_fraction"]
)

display(
    importance_summary
    .sort_values("stability_score", ascending=False)
    .head(50)
)

## Correlated predictors

Do not interpret a single ranked feature as a unique causal effect.

Neighboring Sentinel-2 bands and derived indices are expected to be correlated. If two features contain substitutable information, a tree model may use one in one fold and the other in another fold.

Prioritize:

- importance stable across folds;
- agreement between Random Forest and XGBoost;
- recurrence across functional groups;
- recurrence within meaningful spectral families;
- grouped importance where strong correlation makes individual attribution unstable.

In [ ]:
def feature_family(feature):
    f = feature.lower()

    if f.startswith(("constant_", "intercept_")):
        harmonic = "Intercept"
    elif f.startswith(("amp1_", "amplitude1_")):
        harmonic = "H1 amplitude"
    elif f.startswith(("amp2_", "amplitude2_")):
        harmonic = "H2 amplitude"
    else:
        harmonic = "Other"

    variable = feature.split("_", 1)[1] if "_" in feature else feature
    v = variable.upper()

    if v in {"B2", "B3", "B4"}:
        spectral = "Visible"
    elif v in {"B5", "B6", "B7"}:
        spectral = "Red edge"
    elif v in {"B8", "B8A"}:
        spectral = "NIR"
    elif v in {"B11", "B12"}:
        spectral = "SWIR"
    else:
        spectral = "Index"

    return harmonic, spectral


family_lookup = pd.DataFrame([
    {
        "feature": f,
        "harmonic_family": feature_family(f)[0],
        "spectral_family": feature_family(f)[1],
    }
    for f in IA_COLS
])

importance_with_family = importance_summary.merge(
    family_lookup,
    on="feature",
    how="left",
)

family_importance = (
    importance_with_family
    .groupby(
        [
            "functional_group",
            "model",
            "process",
            "harmonic_family",
            "spectral_family",
        ],
        as_index=False,
    )
    .agg(
        summed_mean_importance=("mean_importance", "sum"),
        mean_stability=("positive_fold_fraction", "mean"),
        n_features=("feature", "nunique"),
    )
    .sort_values("summed_mean_importance", ascending=False)
)

display(family_importance.head(50))

## Important limitation of the family table

The family table above is only a **descriptive aggregation of individual permutation importances**.

The stronger follow-up is **true grouped permutation importance**: permute all members of a correlated information family together in held-out data and measure the performance loss.

Examples:

- all red-edge intercepts together;
- all H1 red-edge amplitudes together;
- all NIR amplitudes together;
- all index intercepts together.

That directly asks whether a correlated information family matters without forcing attribution to one member.

In [ ]:
OUTPUT_DIR = DATA_CSV.parent / "RF_XGBoost_K2_IA_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_results.to_csv(
    OUTPUT_DIR / "model_summary.csv",
    index=False,
)

fold_results.to_csv(
    OUTPUT_DIR / "fold_metrics.csv",
    index=False,
)

importance_results.to_csv(
    OUTPUT_DIR / "fold_permutation_importance.csv",
    index=False,
)

importance_summary.to_csv(
    OUTPUT_DIR / "permutation_importance_summary.csv",
    index=False,
)

family_importance.to_csv(
    OUTPUT_DIR / "family_importance_summary.csv",
    index=False,
)

prediction_results.to_csv(
    OUTPUT_DIR / "oof_predictions.csv",
    index=False,
)

print(f"Results written to: {OUTPUT_DIR}")

## Recommended interpretation sequence

1. Compare Random Forest and XGBoost predictive performance for occurrence, positive abundance, and combined expected cover.
2. Inspect whether feature rankings are stable across folds.
3. Compare rankings between RF and XGBoost.
4. Look for recurring spectral **families**, not just single-band winners.
5. Add true grouped permutation importance for highly correlated families.
6. Use these results alongside PCA/ordination to define candidate clustering feature spaces.
7. Validate candidate spaces by ecological correspondence and cluster stability rather than feature importance alone.